# Drishti — End-to-End App Test (on Colab)

Runs the **real `app/` code** — router, engines, guardrail, translation, speech — against
real photos, on a Colab T4. This is the Phase-1 exit criterion, executed without installing
anything locally.

Everything in `app/` is unit-tested with fakes (117 tests). What has never happened is a
real model loading through it. That is what this notebook checks.

### Why the repo has to be uploaded

The Colab VS Code extension runs *cells* on a Colab machine; it does not copy your project
there. `app/` therefore does not exist on the runtime until we put it there. Cell 1 handles
that.

### Order matters

OCR (PaddlePaddle) and everything else (PyTorch) each bundle an OpenMP runtime, and
co-loading them killed the kernel during the OCR spike with no traceback (`DEC-006`).
`KMP_DUPLICATE_LIB_OK` is set before any import as a mitigation, and the modes are run
**OCR first, VLM last** so that if the process does die you still have the earlier results.

**Runtime → Change runtime type → T4 GPU** before running.

## 1. Get the project onto the runtime

Zip the project locally first (PowerShell, from the folder *above* `drishti`):

```powershell
Compress-Archive -Path drishti -DestinationPath drishti.zip -Force
```

Then run the cell and upload `drishti.zip`. If you later push to GitHub, swap the upload for
`!git clone <url>` — same result, fewer clicks.

In [ ]:
import os

# Set before any framework import -- see DEC-006.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import shutil, sys, zipfile
from pathlib import Path

PROJECT = Path('/content/drishti')

if not PROJECT.exists():
    from google.colab import files
    up = files.upload()                      # choose drishti.zip
    name = next(iter(up))
    with zipfile.ZipFile(name) as z:
        z.extractall('/content')
    # tolerate an archive that wraps everything in one extra folder
    if not PROJECT.exists():
        for cand in Path('/content').glob('*/app/router.py'):
            shutil.move(str(cand.parents[1]), str(PROJECT))
            break

if not (PROJECT / 'app' / 'router.py').exists():
    raise SystemExit(f'{PROJECT}/app/router.py missing -- did the zip contain the drishti folder?')

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print('project root:', PROJECT)
print('modes available:', sorted(p.stem for p in (PROJECT / 'app' / 'modes').glob('[!_]*.py')))

In [ ]:
# The suite needs no models, so a pass here proves the upload is complete and importable
# before we spend minutes downloading weights.
!python -m unittest discover -s tests -t . 2>&1 | tail -4

## 2. Install engines

Weights are **not** downloaded here — every engine loads lazily on first use, so each mode
below pays only for what it needs.

In [ ]:
%pip install -q paddlepaddle paddleocr IndicTransToolkit
print('installed')

## 3. Upload test photos

`test-images/` is gitignored, so the photos are not in the zip. Upload a medicine strip —
and a Marathi/Hindi label too, if you have one, to finally exercise Devanagari Read mode.

In [ ]:
from google.colab import files

IMAGES = Path('/content/photos')
IMAGES.mkdir(exist_ok=True)

for name in files.upload():
    shutil.move(name, IMAGES / name)

photos = sorted(IMAGES.iterdir())
for i, p in enumerate(photos):
    print(f'  [{i}] {p.name}')

STRIP = photos[0]          # <-- change the index if the first upload is not the strip
DEVANAGARI = photos[1] if len(photos) > 1 else None
print('\nstrip     :', STRIP.name)
print('devanagari:', DEVANAGARI.name if DEVANAGARI else '(none uploaded)')

## 4. Medicine mode — the guardrail, end to end

OCR reads the strip, the drug name is matched against the verified database, expiry and MRP
are parsed. If OCR cannot produce a verified name the mode **declines** rather than guessing
(`DEC-007`).

In [ ]:
import time

from app.drug_db import DrugDatabase
from app.engines.paddle_ocr import PaddleOCREngine
from app.modes.medicine import run as run_medicine

ocr = PaddleOCREngine(lang='en')

t0 = time.time()
result = run_medicine(STRIP, ocr, DrugDatabase.from_file())
elapsed = time.time() - t0

print(f'--- medicine mode  ({elapsed:.1f}s) ---')
print('verified :', result.ok)
print('drug     :', result.drug_name)
print('expiry   :', result.expiry_raw, '| expired:', result.expired)
print('MRP      :', result.mrp)
print('\nSPOKEN   :', result.message_en)

if not result.ok:
    print('\nDeclined. Either OCR missed the name, or it is absent from')
    print('data/drug_names_seed.txt -- a 30-entry placeholder, not a real drug database.')

## 5. Marathi output and speech — the Phase-1 exit criterion

In [ ]:
from IPython.display import Audio, display

from app.engines.indictrans import IndicTrans2Translator
from app.engines.mms_tts import MMSTTSEngine
from app.speech import deliver

translator = IndicTrans2Translator()
tts = MMSTTSEngine(out_dir=Path('/content/audio'))

for lang in ('mr', 'hi'):
    t0 = time.time()
    spoken = deliver(result.message_en, lang=lang, translator=translator, tts=tts, speak=True)
    print(f'--- {lang} ({time.time()-t0:.1f}s) ---')
    print(spoken.text_out)
    display(Audio(str(spoken.audio_path)))

## 6. Read mode — Devanagari

`lang='mr'` resolves to `devanagari_PP-OCRv5_mobile_rec`. The code path is confirmed; what
has never been tested is the model against actual Devanagari text.

In [ ]:
from app.modes.read import run as run_read

if DEVANAGARI is None:
    print('No second photo uploaded -- skipping. Read mode in Marathi stays unverified.')
else:
    t0 = time.time()
    text = run_read(DEVANAGARI, PaddleOCREngine(lang='mr'))
    print(f'--- read mode, devanagari ({time.time()-t0:.1f}s) ---')
    print(text)

## 7. Scene mode — the VLM

Last on purpose. This loads PyTorch beside PaddlePaddle, which is the combination that can
abort the process; running it last means a crash costs you nothing already measured.

**If the kernel dies here, that is the documented OpenMP collision, not your mistake.**
Restart, skip to this cell, and it will run on its own.

In [ ]:
from app.engines.smolvlm import SmolVLMEngine
from app.modes.ask import run as run_ask
from app.modes.scene import run as run_scene

vlm = SmolVLMEngine()

t0 = time.time()
print('--- scene mode ---')
print(run_scene(STRIP, vlm), f'({time.time()-t0:.1f}s)')

t0 = time.time()
print('\n--- ask mode ---')
print(run_ask(STRIP, vlm, 'what is written on this?'), f'({time.time()-t0:.1f}s)')

## 8. Findings — fill in, then update `docs/BUILD_PLAN.md`

| Check | Result | Latency |
|---|---|---|
| Medicine: drug name verified | | s |
| Medicine: expiry parsed | | |
| Medicine: MRP parsed | | |
| Marathi translation readable | | s |
| Marathi speech intelligible | | s |
| Hindi speech intelligible | | s |
| Devanagari Read mode | | s |
| Scene mode | | s |
| Kernel survived OCR + VLM together | | |

**Phase 1 is complete when** the medicine row is verified and Marathi audio plays. Tick
those boxes in the build plan and record the latencies against the <8 s target (RISK-1).

Ask a Marathi speaker whether the translation and the synthesized voice are actually
understandable — accuracy metrics do not capture intelligibility, and this is the first time
a human can judge the output.